# Fine Tuning SAM Audio

In [1]:
from anatolian_sam.model_utils import load_sam_audio
from anatolian_sam.latents_dataloader import (
    TurkishMusicLatentsDataset,
    pad_collate_fn,
)
from anatolian_sam.flow_utils import expand_to_256

/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.11.0+cu130)
    Python  3.10.19 (you have 3.11.15)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available

## Load SAM Audio and Processor

In [2]:
base_model, processor = load_sam_audio()

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 41120.63it/s]
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr

## Setup PEFT/LoRA Fine-Tune

In [3]:
import torch

base_model.text_encoder.requires_grad_(False)

with torch.no_grad():
    text_features, text_mask = base_model.text_encoder(["zurna"])

print("Text embeddings shape:", text_features.shape)

Text embeddings shape: torch.Size([1, 4, 768])


### Custom Dataset

In [4]:
train_dataset = TurkishMusicLatentsDataset(
    jsonl_path="../data/train_metadata_tr.jsonl",
    data_base_path="../data/latents",
)

test_dataset = TurkishMusicLatentsDataset(
    jsonl_path="../data/val_metadata_tr.jsonl",
    data_base_path="../data/latents",
)

Loaded 576 audio tuples from train_metadata_tr.jsonl
Loaded 144 audio tuples from val_metadata_tr.jsonl


In [5]:
BATCH_SIZE = 8
ALPHA = 32
RANK = 32
LEARNING_RATE = 5e-5
EPOCHS = 100
DROPOUT = 0.05

In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    collate_fn=pad_collate_fn,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=pad_collate_fn,
    pin_memory=True,
)

In [7]:
for batch in train_loader:
    print("Batch keys:", batch.keys())
    print("Mixture latent shape:", batch["mixture_latent"].shape)
    print("Target latent shape:", batch["target_latent"].shape)
    print("Prompts:", batch["prompt"])
    break  # Just check the first batch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Batch keys: dict_keys(['mixture_latent', 'target_latent', 'target_filename', 'prompt'])
Mixture latent shape: torch.Size([8, 128, 125])
Target latent shape: torch.Size([8, 128, 125])
Prompts: ['bağlama', 'bağlama', 'bağlama', 'bağlama', 'zurna', 'bağlama', 'zurna', 'bağlama']


### Wrap with PEFT/LoRA

In [8]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=RANK,
    lora_alpha=ALPHA,
    # The developers of SAMAudio named the attention layers in their DiT (the DiT handles the actual latent generation) wq and wv
    target_modules=["wq", "wv"],
    lora_dropout=DROPOUT,
    bias="none",
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 8,388,608 || all params: 6,472,553,834 || trainable%: 0.1296


### Initialize Flow Matching Scheduler and Accelerate

In [9]:
from diffusers import FlowMatchEulerDiscreteScheduler
from diffusers.optimization import get_cosine_schedule_with_warmup
from accelerate import Accelerator
from torch.optim import AdamW
import math

# initialize flow-matching scheduler
scheduler = FlowMatchEulerDiscreteScheduler(
    num_train_timesteps=1000,
    shift=1.0,  # standard shift value for flow matching
)

# initialize accelerator for vram and device management
accelerator = Accelerator(gradient_accumulation_steps=4, mixed_precision="bf16")

optimizer = AdamW(peft_model.parameters(), lr=LEARNING_RATE)

# CRITICAL: Because we are accumulating gradients over 4 batches,
# the optimizer only takes a "step" once every 4 forward passes.
num_update_steps_per_epoch = math.ceil(
    len(train_loader) / accelerator.gradient_accumulation_steps
)
max_train_steps = EPOCHS * num_update_steps_per_epoch

# 2. Define Warmup Steps
# A standard rule of thumb for LoRA is warming up for 5% to 10% of total training steps.
num_warmup_steps = int(max_train_steps * 0.10)

# 3. Initialize the Scheduler
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=max_train_steps,
)

# Pass everything to accelerator.prepare
peft_model, optimizer, train_loader, test_loader, lr_scheduler = accelerator.prepare(
    peft_model, optimizer, train_loader, test_loader, lr_scheduler
)

print("Flow-Matching Scheduler and Accelerator Initialized!")
print(f"Total Training Steps: {max_train_steps}")
print(f"Warmup Steps: {num_warmup_steps}")

Flow-Matching Scheduler and Accelerator Initialized!
Total Training Steps: 1800
Warmup Steps: 180


## Custom LoRA Flow-Matching Training Loop

In [10]:
import os
import torch
import torch.nn.functional as F
import wandb
from tqdm.autonotebook import tqdm

In [11]:
import inspect

# Inspect the forward method of the underlying base model
signature = inspect.signature(base_model.forward)

print("SAM Audio Forward Signature:")
for param in signature.parameters.values():
    print(f"- {param.name}: {param.default}")

SAM Audio Forward Signature:
- noisy_audio: <class 'inspect._empty'>
- audio_features: <class 'inspect._empty'>
- text_features: <class 'inspect._empty'>
- time: <class 'inspect._empty'>
- masked_video_features: None
- text_mask: None
- anchor_ids: None
- anchor_alignment: None
- audio_pad_mask: None


In [12]:
wandb.init(
    entity="zeerafle-sivas-cumhuriyet-university",
    project="turkish-sam-audio",
    name="higher-rank-lora-flow-matching-run",
    config={
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "lora_rank": RANK,
        "lora_alpha": ALPHA,
        "lora_dropout": DROPOUT,
    },
    dir="../wandb",
)

# Setup Checkpointing Directory
save_directory = "../checkpoints/turkish_sam_audio_lora_best"
os.makedirs(save_directory, exist_ok=True)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: zeerafle (zeerafle-sivas-cumhuriyet-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
def train_epoch(
    peft_model,
    base_model,
    train_loader,
    optimizer,
    lr_scheduler,
    accelerator,
    epoch,
    epochs,
    global_step,
):
    """Handles a single training epoch."""
    peft_model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs - 1}")

    for step, batch in enumerate(pbar):
        # 1. Encode Text (No Gradients)
        with torch.no_grad():
            text_features, text_mask = base_model.text_encoder(batch["prompt"])
            text_features = text_features.to(accelerator.device)
            text_mask = text_mask.to(accelerator.device)

        # 2. Forward & Backward Pass (with Gradient Accumulation)
        with accelerator.accumulate(peft_model):
            mixture_latent = batch["mixture_latent"].transpose(1, 2)
            target_latent = batch["target_latent"].transpose(1, 2)

            # Time Sampling
            t = torch.rand((mixture_latent.size(0),), device=accelerator.device)
            t_expanded = t.view(-1, 1, 1)

            # Meta's 256-dim expansion trick
            mixture_256 = expand_to_256(mixture_latent)
            target_256 = expand_to_256(target_latent)

            # Generate Noise and Noisy State
            noise_256 = torch.randn_like(target_256)
            x_t_256 = (1 - t_expanded) * noise_256 + t_expanded * target_256

            with accelerator.autocast():
                predicted_velocity = peft_model(
                    noisy_audio=x_t_256,
                    time=t,
                    audio_features=mixture_256,
                    text_features=text_features,
                    text_mask=text_mask,
                )

                # Flow-matching Vector Field Discrepancy Loss
                true_velocity = target_256 - noise_256
                loss = F.mse_loss(predicted_velocity, true_velocity)

            # Update weights
            accelerator.backward(loss)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        # 3. Logging & Progress tracking
        pbar.set_postfix(loss=f"{loss.item():.4f}")
        wandb.log(
            {
                "train/step_loss": loss.item(),
                "train/learning_rate": lr_scheduler.get_last_lr()[0],
                "global_step": global_step,
            }
        )

        epoch_loss += loss.item()
        global_step += 1

    return epoch_loss / len(train_loader), global_step

In [14]:
def validate_epoch(peft_model, base_model, test_loader, accelerator, epoch, epochs):
    """Handles a single validation epoch."""
    peft_model.eval()
    val_loss = 0.0
    val_pbar = tqdm(test_loader, desc=f"Val Epoch {epoch}/{epochs - 1}", colour="green")

    with torch.no_grad():
        for step, batch in enumerate(val_pbar):
            text_features, text_mask = base_model.text_encoder(batch["prompt"])
            text_features = text_features.to(accelerator.device)
            text_mask = text_mask.to(accelerator.device)

            mixture_latent = batch["mixture_latent"].transpose(1, 2)
            target_latent = batch["target_latent"].transpose(1, 2)

            t = torch.rand((mixture_latent.size(0),), device=accelerator.device)
            t_expanded = t.view(-1, 1, 1)

            mixture_256 = expand_to_256(mixture_latent)
            target_256 = expand_to_256(target_latent)

            noise_256 = torch.randn_like(target_256)
            x_t_256 = (1 - t_expanded) * noise_256 + t_expanded * target_256

            with accelerator.autocast():
                predicted_velocity = peft_model(
                    noisy_audio=x_t_256,
                    time=t,
                    audio_features=mixture_256,
                    text_features=text_features,
                    text_mask=text_mask,
                )
                true_velocity = target_256 - noise_256
                v_loss = F.mse_loss(predicted_velocity, true_velocity)

            val_loss += v_loss.item()
            val_pbar.set_postfix(loss=f"{v_loss.item():.4f}")

    return val_loss / len(test_loader)

In [15]:
# Early Stopping Configuration
patience = 7
min_delta = 0.0001
patience_counter = 0
best_val_loss = float("inf")
global_step = 0
save_directory = "../checkpoints/best_lora"

for epoch in range(EPOCHS):
    # Run Training
    avg_train_loss, global_step = train_epoch(
        peft_model,
        base_model,
        train_loader,
        optimizer,
        lr_scheduler,
        accelerator,
        epoch,
        EPOCHS,
        global_step,
    )

    # Run Validation
    avg_val_loss = validate_epoch(
        peft_model, base_model, test_loader, accelerator, epoch, EPOCHS
    )

    # Log Epoch Metrics
    wandb.log(
        {
            "train/epoch_loss": avg_train_loss,
            "val/epoch_loss": avg_val_loss,
            "epoch": epoch,
        }
    )

    print(
        f"--- Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} ---"
    )

    # Early Stopping & Artifact Saving Logic
    if avg_val_loss < (best_val_loss - min_delta):
        print(
            f"✅ Validation loss improved ({best_val_loss:.4f} -> {avg_val_loss:.4f}). Saving checkpoint..."
        )
        best_val_loss = avg_val_loss
        patience_counter = 0

        accelerator.wait_for_everyone()

        # Save locally and push to W&B Artifacts
        if accelerator.is_main_process:
            unwrapped_model = accelerator.unwrap_model(peft_model)
            unwrapped_model.save_pretrained(save_directory, safe_serialization=True)

            # --- W&B ARTIFACT SAVING ---
            # Creates an artifact version in your wandb dashboard
            model_artifact = wandb.Artifact(
                name=f"sam-audio-lora-run-{wandb.run.id}",
                type="model",
                description=f"Best model at epoch {epoch} with val_loss {best_val_loss:.4f}",
            )
            model_artifact.add_dir(save_directory)
            wandb.log_artifact(model_artifact)

    else:
        patience_counter += 1
        print(
            f"⚠️ Validation loss did not improve. Patience: {patience_counter}/{patience}"
        )

        if patience_counter >= patience:
            print(
                "🛑 Early stopping triggered! Training halted to prevent overfitting."
            )
            break


Epoch 0/99:   0%|          | 0/72 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Val Epoch 0/99:   0%|       

--- Epoch 0 | Train Loss: 0.6589 | Val Loss: 0.5463 ---
✅ Validation loss improved (inf -> 0.5463). Saving checkpoint...


wandb: Adding directory to artifact (../checkpoints/best_lora)... Done. 0.2s
Epoch 1/99:   0%|          | 0/72 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the

--- Epoch 1 | Train Loss: 0.6339 | Val Loss: 0.5401 ---
✅ Validation loss improved (0.5463 -> 0.5401). Saving checkpoint...


...

--- Epoch 32 | Train Loss: 0.3098 | Val Loss: 0.2562 ---
⚠️ Validation loss did not improve. Patience: 5/7


Epoch 33/99:   0%|          | 0/72 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARA

--- Epoch 33 | Train Loss: 0.3111 | Val Loss: 0.2781 ---
⚠️ Validation loss did not improve. Patience: 6/7


Epoch 34/99:   0%|          | 0/72 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARA

--- Epoch 34 | Train Loss: 0.3077 | Val Loss: 0.2685 ---
⚠️ Validation loss did not improve. Patience: 7/7
🛑 Early stopping triggered! Training halted to prevent overfitting.


In [16]:
# 1. Ensure # 1. Ensure all GPUs have finished their final calculations
accelerator.wait_for_everyone()

# 2. Define a DISTINCT output directory for the final epoch
# Notice the "_final" suffix to prevent overwriting your "_best" checkpoint
save_directory = "../checkpoints/final_lora"
os.makedirs(save_directory, exist_ok=True)

# 3. Unwrap and Save the final epoch state
if accelerator.is_main_process:
    unwrapped_model = accelerator.unwrap_model(peft_model)
    unwrapped_model.save_pretrained(
        save_directory,
        safe_serialization=True,
    )

    model_artifact = wandb.Artifact(
        name=f"sam-audio-lora-run-{wandb.run.id}-final",
        type="model",
        description=f"Final model at epoch {epoch} with val_loss {best_val_loss:.4f}",
    )
    model_artifact.add_dir(save_directory)
    wandb.log_artifact(model_artifact)

print(f"Final epoch LoRA adapters successfully saved to: {save_directory}")

wandb.finish()

wandb: Adding directory to artifact (../checkpoints/final_lora)... Done. 0.2s


Final epoch LoRA adapters successfully saved to: ../checkpoints/final_lora


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
global_step,▁▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇███
train/epoch_loss,██▇▆▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁
train/learning_rate,▁▂▂▂▅▅▅▆▆▇████████████████████▇▇▇▇▇▇▇▇▇▇
train/step_loss,██▇▆▄▄▆▅▄▄▂▅▃▃▂▅▂▄▄▄▂▂▂▂▃▇▃▃▃▂▂▃▁▂▁▃▃▂▂▃
val/epoch_loss,██▇▆▅▆▄▄▄▄▃▃▃▃▃▂▃▂▂▂▂▂▂▂▁▁▂▁▂▂▁▁▁▂▁
epoch,34
global_step,2519
train/epoch_loss,0.30773
train/learning_rate,4e-05
train/step_loss,0.21269
